# Satellite PM2.5 Model Training

This notebook trains baseline PM2.5 regression models using the preprocessed two-station satellite dataset in `data/processed/satellite_pm25_daily`.

The workflow uses chronological train, validation, and test splits that were already created during preprocessing. Model selection is based on validation RMSE, then all models are evaluated on the held-out test split.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.append(str(PROJECT_ROOT / 'scripts'))
from train_satellite_pm25_models import DATA_DIR, OUTPUT_DIR, train_and_evaluate

DATA_DIR, OUTPUT_DIR

(WindowsPath('data/processed/satellite_pm25_daily'),
 WindowsPath('outputs/satellite_pm25_models'))

In [2]:
metrics, predictions, best_model_name = train_and_evaluate()
print(f'Best model by validation RMSE: {best_model_name}')
metrics.sort_values(['split', 'rmse'])

FileNotFoundError: [Errno 2] No such file or directory: 'data\\processed\\satellite_pm25_daily\\train.csv'

## Latest Run Result Log

Latest run generated by `python scripts/train_satellite_pm25_models.py`.

Best model by validation RMSE: `ridge`

Validation metrics:

| model | n_rows | MAE | RMSE | R2 |
| --- | ---: | ---: | ---: | ---: |
| ridge | 121 | 33.2630 | 61.5368 | 0.3471 |
| random_forest | 121 | 40.2968 | 65.8999 | 0.2512 |
| gradient_boosting | 121 | 40.0067 | 67.6876 | 0.2100 |
| extra_trees | 121 | 38.7057 | 72.0956 | 0.1038 |
| dummy_mean | 121 | 43.9928 | 76.5528 | -0.0105 |

Test metrics:

| model | n_rows | MAE | RMSE | R2 |
| --- | ---: | ---: | ---: | ---: |
| gradient_boosting | 121 | 29.3003 | 102.9442 | 0.0236 |
| dummy_mean | 121 | 28.6550 | 104.1895 | -0.0001 |
| ridge | 121 | 39.4591 | 104.4237 | -0.0046 |
| extra_trees | 121 | 24.8729 | 105.1878 | -0.0194 |
| random_forest | 121 | 28.8330 | 105.6042 | -0.0275 |

Note: validation selects `ridge`, but test performance is weak for all models and close to the mean baseline. This suggests the current features and small two-station split have limited ability to generalize to the later test period.

In [ ]:
import pandas as pd

metrics = pd.read_csv(OUTPUT_DIR / 'metrics.csv')
predictions = pd.read_csv(OUTPUT_DIR / 'predictions.csv', parse_dates=['date'])

display(metrics.sort_values(['split', 'rmse']))
display(predictions.head())

## Visualizations

In [ ]:
from IPython.display import Image, display

plot_files = [
    'model_comparison_rmse.png',
    'observed_vs_predicted_test.png',
    'test_residuals_by_date.png',
    'test_timeseries_station_2161292.png',
    'test_timeseries_station_2161306.png',
    'feature_importance.png',
]

for file_name in plot_files:
    path = OUTPUT_DIR / file_name
    if path.exists():
        print(file_name)
        display(Image(filename=str(path)))